[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Memory Hierarchy and Cache** {#memory-hierarchy-and-cache}

Chapter 07 treated the instruction-fetch and memory-access stages as if an instruction or data word normally arrived in one cycle. That assumption is essential for the ideal five-stage pipeline, but a large main memory cannot usually meet the processor's shortest clock period. If every IF request and every load waited directly for DRAM, the pipeline would spend far more time stalled than executing.

A **cache** is a small, fast structure that stores copies of selected memory blocks. It is transparent to ordinary program semantics: a load must return the same architectural value whether it hits in L1, hits in a lower cache, or eventually reaches DRAM. The cache changes where and when the value is found, not the value that software is entitled to observe.

This chapter studies physical cache organization and its performance consequences. Address translation, page tables, and TLBs are reserved for Chapter 09. Until then, each cache example uses a physical address, or equivalently assumes that translation has already supplied the address used for lookup.

### **Why Computers Need a Memory Hierarchy** {#why-computers-need-a-memory-hierarchy}

Memory design asks for conflicting properties. The processor wants low latency so a dependent instruction can continue quickly, high bandwidth so several instructions can be supplied at once, large capacity for active programs, low energy, and low cost per bit. No single storage technology optimizes all of them simultaneously.

![Registers and several cache levels keep selected blocks near the CPU, while larger and usually slower levels provide capacity. A miss travels downward and a returned block can populate upper levels.](assets/memory-hierarchy-tradeoffs.svg){fig-align="center" width="100%"}

The hierarchy works because upper levels do not need to contain all memory. They need to contain the **working set** that is likely to be used soon. Registers hold current operands, split L1 instruction and data caches serve the pipeline's two request streams, lower caches retain larger recent working sets, and DRAM supplies much greater capacity. Persistent storage sits below main memory in a complete system, but it is not an ordinary cache-hit path for each load.

| Level | Primary design goal | Typical consequence of a miss |
|---|---|---|
| registers | immediate operand availability | instructions must move or recompute a value |
| L1I / L1D | shortest common hit path and high local bandwidth | request continues to a lower cache |
| L2 / last-level cache | reduce expensive main-memory traffic | longer on-chip lookup or DRAM request |
| DRAM | large volatile working capacity | operating-system or storage activity may be required if the page is absent |

The hierarchy therefore optimizes the common case: most references should terminate near the processor, while the less frequent misses pay the longer path. A cache with a very short hit time but poor hit rate may still perform badly, and a large cache that slows the processor clock may lose more than it saves. Capacity, associativity, block size, replacement, and write policy all interact.

### **Temporal and Spatial Locality** {#temporal-and-spatial-locality}

**Locality of reference** is the empirical tendency of programs to use a limited region of instructions and data during a short interval. It is not guaranteed by the ISA. It arises from loops, procedure calls, stack use, sequential instruction fetch, array traversal, and repeated access to active objects.

![Temporal locality is reuse of the same block within a short time. Spatial locality is use of nearby addresses from a transferred block.](assets/temporal-spatial-locality.svg){fig-align="center" width="100%"}

**Temporal locality** means a recently referenced item is likely to be referenced again. A loop repeatedly executes the same instruction block and updates the same counter. Retaining those blocks lets later references hit. **Spatial locality** means an address near a recent reference is likely to be used. Fetching one aligned block brings neighboring instructions or array elements before they are individually requested.

The useful unit is the block, not the source-language variable. If a 64-byte block contains eight 8-byte integers, sequential traversal can obtain eight elements from one lower-level transfer. A stride of 64 bytes touches one element per block and uses none of the other transferred bytes. A block that is spatially useful but never reused still benefits from one transfer; a repeatedly reused block demonstrates temporal locality.

**Reuse distance** counts the distinct blocks referenced between two accesses to the same block. A small reuse distance suggests that a modest cache can retain the block. When the reuse distance exceeds the effective number of available lines in its set or working cache, the block is likely to be evicted before reuse.

<details>
<summary>Python example: compare sequential and strided block footprints</summary>

```python
def block_trace(addresses, block_size):
    """Convert byte addresses into the memory-block numbers a cache sees."""
    return [address // block_size for address in addresses]


base = 0x1000
element_size = 8
block_size = 64

# Eight neighboring 64-bit values all occupy one 64-byte block.
sequential = [base + i * element_size for i in range(8)]

# Eight values separated by one full block occupy eight different blocks.
strided = [base + i * block_size for i in range(8)]

sequential_blocks = block_trace(sequential, block_size)
strided_blocks = block_trace(strided, block_size)

assert len(set(sequential_blocks)) == 1
assert len(set(strided_blocks)) == 8
print("sequential blocks:", sequential_blocks)
print("strided blocks:   ", strided_blocks)
```

</details>

Locality explains why caching works, but it also explains when caching fails. Streaming through a data set much larger than the cache provides spatial locality but little temporal reuse. Alternating between addresses forced into the same set can destroy locality through conflicts even when total cache capacity is available.

### **Cache Blocks and Address Decomposition** {#cache-blocks-and-address-decomposition}

A cache stores a memory region as fixed-size **blocks**, also called cache lines. A block represents a contiguous, naturally aligned range, usually with a power-of-two size. If the block size is $B$ bytes, byte address $A$ belongs to

$$
BlockNumber=\left\lfloor\frac{A}{B}\right\rfloor,
\qquad
BlockOffset=A\bmod B.
$$

- $A$ is the requested byte address.
- $B$ is the number of bytes copied and managed together.
- `BlockNumber` identifies the aligned memory region.
- `BlockOffset` identifies the requested byte inside that region.

A set-associative cache divides its data capacity into sets and ways. Let $C$ be data capacity in bytes, $B$ block size, and $E$ associativity or ways per set. The number of sets is

$$
S=\frac{C}{B\times E}.
$$

For power-of-two dimensions and an $m$-bit address,

$$
b=\log_2 B,\qquad s=\log_2 S,\qquad t=m-s-b,
$$

where $b$, $s$, and $t$ are the block-offset, set-index, and tag bit counts. The example used throughout this chapter has $C=32$ KiB, $B=64$ bytes, $E=4$, and $m=32$. It therefore has 128 sets, a 6-bit offset, a 7-bit index, and a 19-bit tag.

::: {.diagram-scroll .wide-diagram}
![Address 0x1234ABCD selects set 47 in a 32 KiB four-way cache. Four tags are compared, the valid matching way produces a hit, and offset 13 selects bytes from that 64-byte block.](assets/cache-address-lookup.svg){fig-align="center"}
:::

<details>
<summary>Python example: calculate cache geometry and decompose an address</summary>

```python
from math import log2


def cache_fields(address, address_bits, capacity_bytes, block_bytes, ways):
    """Return geometry and tag/index/offset values for a power-of-two cache."""
    sets = capacity_bytes // (block_bytes * ways)
    offset_bits = int(log2(block_bytes))
    index_bits = int(log2(sets))
    tag_bits = address_bits - index_bits - offset_bits

    offset = address & (block_bytes - 1)
    index = (address >> offset_bits) & (sets - 1)
    tag = address >> (offset_bits + index_bits)

    return {
        "sets": sets,
        "tag_bits": tag_bits,
        "index_bits": index_bits,
        "offset_bits": offset_bits,
        "tag": tag,
        "index": index,
        "offset": offset,
    }


fields = cache_fields(
    address=0x1234ABCD,
    address_bits=32,
    capacity_bytes=32 * 1024,
    block_bytes=64,
    ways=4,
)

assert fields == {
    "sets": 128,
    "tag_bits": 19,
    "index_bits": 7,
    "offset_bits": 6,
    "tag": 0x91A5,
    "index": 47,
    "offset": 13,
}
print(fields)
```

</details>

The advertised cache capacity normally counts data bytes, not tags, valid bits, dirty bits, error-correction bits, and replacement metadata. Those structures consume additional area and energy, which is one reason increasing associativity is not free.

#### **Tag** {#tag}

The **tag** stores the high-order address identity of the memory block occupying a way. The index can select a set, but many different memory blocks map to that same set. The tag distinguishes which one is currently present.

For way $w$, a read hit is conceptually

$$
Hit_w=Valid_w\land(Tag_w=RequestedTag).
$$

- $Valid_w$ is false after reset or invalidation, when the line contains no usable block.
- $Tag_w$ is the identity stored beside the data block in way $w$.
- `RequestedTag` comes from the high address bits.
- A set-associative hit is the OR of the per-way hit signals; at most one valid way should match.

The valid bit matters because arbitrary SRAM power-up contents can accidentally equal a requested tag. A dirty bit has a different role: it records whether a write-back line contains changes not yet propagated downward. Dirty state affects eviction, not whether the requested block is present.

Larger addresses or smaller blocks often increase tag storage. More ways require more tags to be read and compared in parallel. Designers may stage or predict these operations to protect hit latency, but the basic identity check remains unchanged.

#### **Index** {#index}

The **index** selects exactly one cache set. In a power-of-two cache, it is a contiguous field immediately above the block offset. Equivalently,

$$
SetIndex=BlockNumber\bmod S.
$$

For the running example, $S=128$, so index 47 selects the four tag/data entries belonging to set 47. Only those four ways can contain the requested block. Other sets need not be activated, saving lookup work compared with comparing every cache line.

The index creates placement restrictions. Blocks whose numbers differ by a multiple of $S$ share an index. If they are active at the same time, they compete for the $E$ ways of one set even if other sets are empty. Such collisions cause conflict misses.

Indexing is fast because it behaves like array selection, but the selected set must be ready early enough for tag comparison and data selection to fit the cache hit path. Chapter 09 will explain how address translation interacts with this timing; this chapter deliberately uses the already-selected cache address.

#### **Block Offset** {#block-offset}

The **block offset** is formed by the lowest $b=\log_2B$ address bits. It chooses a byte or word inside the matching block and does not decide which block is cached. With 64-byte blocks, the offset has six bits and ranges from 0 to 63.

Address `0x1234ABCD` has offset 13, so it requests byte 13 of the line beginning at the aligned address

$$
BlockBase=A-(A\bmod B)=\texttt{0x1234ABC0}.
$$

The cache may read an entire way-sized block or a bank while tags are being checked, then use the offset and access size to select the requested bytes. A load that spans a block boundary may require two line accesses because its bytes have different tags and possibly different sets.

Block size exposes a trade-off. Larger blocks exploit more spatial locality and reduce the number of tag entries for fixed capacity. They also take longer to transfer, consume more bandwidth on a miss, reduce the number of independently retained blocks, and may fetch bytes that are never used.

### **Cache Mapping** {#cache-mapping}

**Mapping** defines where a memory block is allowed to reside. A cache with $S$ sets and $E$ ways has $S\times E$ data lines. Each block maps to one set, then may occupy any way in that set. Direct mapping, set associativity, and full associativity are therefore points on one continuum rather than unrelated structures.

::: {.diagram-scroll .wide-diagram}
![A four-line direct-mapped cache has four one-way sets, a two-way cache has two two-way sets, and a fully associative cache has one four-way set. Placement freedom rises with associativity.](assets/cache-mapping-comparison.svg){fig-align="center"}
:::

| Organization | Sets | Ways per set | Tag comparisons per selected set | Replacement choice |
|---|---:|---:|---:|---|
| direct-mapped | number of lines | 1 | 1 | none; the indexed line is forced |
| $E$-way set-associative | lines / $E$ | $E$ | $E$ | choose one way within the set |
| fully associative | 1 | number of lines | every line | choose any line |

Increasing associativity generally reduces placement conflicts, but increases tag-comparison work, way-selection logic, replacement metadata, and often energy. The best choice depends on access patterns and the latency budget of that cache level.

#### **Direct-Mapped Cache** {#direct-mapped-cache}

A **direct-mapped cache** has one way per set. Memory block $n$ must use

$$
line=n\bmod L,
\qquad
tag=\left\lfloor\frac{n}{L}\right\rfloor,
$$

where $L$ is the number of cache lines. Only one tag comparison and no replacement policy are required. This produces a short, simple hit path.

The restriction also creates ping-pong behavior. In a four-line cache, blocks 0 and 4 both map to line 0. The trace `0, 4, 0, 4` misses every time because each access evicts the block needed next, although lines 1 through 3 remain available.

<details>
<summary>Python model: trace a direct-mapped cache</summary>

```python
def direct_mapped_trace(blocks, line_count):
    """Return hit/miss decisions and final tag state."""
    tags = [None] * line_count
    events = []

    for block in blocks:
        line = block % line_count
        tag = block // line_count
        hit = tags[line] == tag
        events.append((block, line, tag, "hit" if hit else "miss"))
        if not hit:
            tags[line] = tag

    return events, tags


events, tags = direct_mapped_trace([0, 4, 0, 4], line_count=4)
assert [event[-1] for event in events] == ["miss"] * 4
assert tags == [1, None, None, None]  # block 4 remains in line 0
print(*events, sep="\n")
```

</details>

Direct mapping can still be an effective design when the hit-time advantage outweighs conflict costs, particularly for small latency-critical structures or workloads with benign placement.

#### **Set-Associative Cache** {#set-associative-cache}

An **$E$-way set-associative cache** lets a block occupy any of $E$ ways in its indexed set. A two-way cache allows two colliding blocks to coexist; a four-way cache allows four. The set is still selected by $BlockNumber\bmod S$, so only its ways are searched.

On a hit, all way tags are normally compared in parallel and a way multiplexer selects the matching data. On a miss, an invalid way is preferred. If all ways are valid, replacement policy selects a victim. The running 32 KiB cache has 128 sets and four ways, so each set can retain four blocks that share the same seven-bit index.

Set associativity is the common compromise between direct mapping's short lookup and full associativity's placement freedom. Higher associativity reduces conflict misses with diminishing returns, while the comparator, multiplexer, and replacement network become more expensive.

#### **Fully Associative Cache** {#fully-associative-cache}

A **fully associative cache** has one set containing every line. There is no index field; the entire block identity acts as the tag. Any incoming block can occupy any line, so placement restrictions do not create conflict misses. Capacity and compulsory misses still remain.

The cost is associative search across all valid tags and a replacement choice across the whole structure. Full associativity is therefore most practical for small structures or special buffers rather than large first-level data arrays.

<details>
<summary>Python model: compare one-, two-, and four-way placement</summary>

```python
class SetAssociativeCache:
    """Small LRU cache model using block numbers rather than byte addresses."""

    def __init__(self, line_count, ways):
        if line_count % ways != 0:
            raise ValueError("ways must divide line_count")
        self.ways = ways
        self.sets = [[] for _ in range(line_count // ways)]

    def access(self, block):
        entries = self.sets[block % len(self.sets)]
        if block in entries:
            entries.remove(block)
            entries.insert(0, block)  # most recently used first
            return True
        if len(entries) == self.ways:
            entries.pop()            # evict least recently used
        entries.insert(0, block)
        return False


trace = [0, 4, 0, 4]
misses = {}

for ways, label in [(1, "direct"), (2, "two-way"), (4, "fully")]:
    cache = SetAssociativeCache(line_count=4, ways=ways)
    misses[label] = sum(not cache.access(block) for block in trace)

assert misses == {"direct": 4, "two-way": 2, "fully": 2}
print(misses)
```

</details>

The two blocks communicate the same data under every organization. Associativity changes only whether both copies can remain resident at once and how much hardware is required to find them.

### **Replacement Policies** {#replacement-policies}

A **replacement policy** chooses which valid line to evict after a miss selects a full set. Direct-mapped caches have no choice; the indexed line is forced. Associative caches first use an invalid way if one exists, because filling empty capacity loses no cached block.

![For trace A, B, C, A, D in a three-way set, LRU evicts B, FIFO evicts A, and random may choose any way. Hits update LRU recency but not FIFO insertion order.](assets/cache-replacement-policies.svg){fig-align="center" width="100%"}

| Policy | State used | Intuition | Limitation |
|---|---|---|---|
| LRU | recency order | a recently used block may be reused soon | exact state grows costly with many ways |
| pseudo-LRU | compressed direction/state bits | approximate LRU with less hardware | can choose a non-LRU victim |
| FIFO | insertion order | remove the oldest resident | ignores reuse on hits |
| random | pseudo-random way | avoid ordering metadata and systematic patterns | can evict a hot line |
| optimal/Belady | future trace | evict the block used farthest in the future | impossible online; useful as a bound |

<details>
<summary>Python example: compare LRU and FIFO on the same set trace</summary>

```python
from collections import deque


def replacement_trace(accesses, capacity, policy):
    """Return the resident order and evicted block after each access."""
    resident = deque()
    events = []

    for block in accesses:
        evicted = None
        hit = block in resident

        if hit and policy == "LRU":
            resident.remove(block)
            resident.appendleft(block)
        elif not hit:
            if len(resident) == capacity:
                evicted = resident.pop()
            resident.appendleft(block)

        events.append((block, hit, evicted, tuple(resident)))

    return events


trace = list("ABCAD")
lru = replacement_trace(trace, capacity=3, policy="LRU")
fifo = replacement_trace(trace, capacity=3, policy="FIFO")

assert lru[-1][2] == "B"
assert fifo[-1][2] == "A"
print("LRU final:", lru[-1])
print("FIFO final:", fifo[-1])
```

</details>

Replacement is a prediction problem: the controller tries to retain blocks that will be useful before their next eviction opportunity. No online policy dominates every access pattern, so implementation cost and workload behavior matter as much as the policy name.

### **Write Policies** {#write-policies}

Loads only need to locate a current copy. Stores create a second responsibility: the hierarchy must determine which copy changes now, when lower levels receive the new value, and what happens when the target block is absent. Two independent choices answer those questions.

::: {.diagram-scroll .wide-diagram}
![Write-through and write-back govern propagation after a hit. Write-allocate and no-write-allocate govern whether a store miss fills the cache.](assets/cache-write-policies.svg){fig-align="center"}
:::

The policies operate at block granularity even when a store changes only one byte. Byte-enable signals update selected bytes in the cache line. A write-back cache must preserve all other bytes and eventually transfer a complete or supported partial update to the lower level.

#### **Write-Through and Write-Back** {#write-through-and-write-back}

With **write-through**, every store hit updates both the cache and the next lower level. The lower level therefore receives changes promptly, and an evicted line need not be written merely because of that store. The cost is lower-level traffic for every store. A write buffer can decouple the pipeline from some of that latency, but the buffer can fill and force stalls.

With **write-back**, a store hit updates only the cache line and sets its dirty bit. Lower memory may temporarily contain an older value. When the dirty line is evicted, the cache first writes it to the next level, then installs the replacement. Multiple stores to one resident line can therefore collapse into one write-back transfer.

$$
EvictionCost=
\begin{cases}
WriteBlock+ReadReplacement, & dirty=1,\\
ReadReplacement, & dirty=0.
\end{cases}
$$

The terms represent transfers rather than universal cycle counts. Their actual latency depends on lower-level queues, bandwidth, and whether writes can overlap reads.

<details>
<summary>Python model: repeated stores under write-through and write-back</summary>

```python
def repeated_store_traffic(store_count, policy, evict_after=True):
    """Count lower-level writes for repeated hits to one resident line."""
    dirty = False
    lower_writes = 0

    for _ in range(store_count):
        if policy == "write-through":
            lower_writes += 1
        elif policy == "write-back":
            dirty = True
        else:
            raise ValueError("unknown policy")

    if policy == "write-back" and dirty and evict_after:
        lower_writes += 1
        dirty = False

    return lower_writes, dirty


assert repeated_store_traffic(3, "write-through") == (3, False)
assert repeated_store_traffic(3, "write-back") == (1, False)
print("three stores: 3 lower writes versus 1 write-back on eviction")
```

</details>

Write-back reduces traffic but makes dirty-line state part of correctness. A fault, flush, coherence request, or cache-block management operation must not silently discard the only current copy.

#### **Write-Allocate and No-Write-Allocate** {#write-allocate-and-no-write-allocate}

On a store miss, **write-allocate** fetches the containing block, installs it, and then applies the store. This pays a read-for-ownership style miss cost but makes subsequent accesses to nearby bytes likely to hit. It pairs naturally with write-back when stores exhibit locality.

**No-write-allocate**, also called write-around, sends the store to the lower level without installing the missing block in this cache. It avoids filling the cache with a line that may never be read again, which can help streaming writes. A later load to the same address will still miss unless another mechanism brought the block in.

| Expected behavior after the miss | More suitable tendency |
|---|---|
| repeated stores or nearby reads | write-allocate retains the block |
| one-pass streaming output | no-write-allocate avoids cache pollution |
| partial-byte store with write-back | allocate so unchanged bytes are retained correctly |
| abundant write-buffer bandwidth | write-through/no-write-allocate becomes easier to tolerate |

Write-through plus no-write-allocate and write-back plus write-allocate are common pairings, not architectural laws. Each cache level may use different policies, and special non-temporal operations can request behavior intended for streaming data.

### **Cache Misses** {#cache-misses}

A cache **hit** occurs when the selected set contains a valid matching tag. A **miss** means the requested block is not available in that cache and the request must continue downward. The miss penalty can include lower-level lookup, dirty victim write-back, block transfer, installation, and restart of the waiting access.

The classic **3C model** classifies misses by comparing the target cache with idealized reference caches. The categories explain different causes and therefore suggest different remedies.

![A first reference is compulsory. A working set larger than an equal-size fully associative cache causes capacity misses. Limited set placement creates conflict misses.](assets/cache-miss-three-c.svg){fig-align="center" width="100%"}

The classification is conceptual rather than a field stored by ordinary cache hardware. Performance tools or simulators classify a trace by tracking first references and comparing multiple cache organizations.

#### **Compulsory Misses** {#compulsory-misses}

A **compulsory miss**, or cold miss, is the first reference to a block in the measured execution. No initially empty cache could already contain it. After the block is fetched, later references may hit if replacement and placement preserve it.

Larger blocks can reduce the number of demand misses when the program soon accesses neighboring bytes, because one compulsory transfer covers several future references. The trade-off is extra bandwidth, longer refill, and fewer independent lines. Hardware or software prefetching can move the compulsory transfer before the demand, converting visible latency into speculative traffic rather than making the first transfer disappear.

Compulsory misses are sensitive to the measurement boundary. A warmed cache may already contain blocks from earlier execution, while a cold-cache experiment intentionally begins empty. Meaningful comparisons must state which condition is used.

#### **Capacity Misses** {#capacity-misses}

A **capacity miss** occurs because the total active working set exceeds cache capacity. It would still occur in a fully associative cache of the same line count and block size under the reference replacement policy. Placement freedom cannot help because too many distinct useful blocks compete for too few total lines.

The trace `A, B, C, D, A` has a capacity miss on the final `A` in a three-line fully associative cache: four distinct blocks became active before reuse. Increasing associativity without adding lines cannot make all four fit.

Capacity misses can be reduced by increasing data capacity, shortening reuse distance, or processing a smaller working set at a time. Loop tiling is a software example: it completes more work on a cache-sized region before moving to another region.

#### **Conflict Misses** {#conflict-misses}

A **conflict miss** occurs because the target cache's set-placement restriction evicts a useful block that could remain in a fully associative cache of the same total capacity. The direct-mapped `0, 4, 0, 4` example is conflict-dominated: both blocks insist on line 0 while other lines are unused.

Higher associativity, a victim cache, alternative indexing, or a changed data alignment can reduce conflicts. Larger total capacity may also change index bits and incidentally separate colliding addresses, but the defining cause is placement rather than total line count.

<details>
<summary>Python model: classify a trace with the 3C method</summary>

```python
class LRUBlockCache:
    def __init__(self, line_count, ways):
        self.ways = ways
        self.sets = [[] for _ in range(line_count // ways)]

    def access(self, block):
        entries = self.sets[block % len(self.sets)]
        hit = block in entries
        if hit:
            entries.remove(block)
        elif len(entries) == self.ways:
            entries.pop()
        entries.insert(0, block)
        return hit


def classify_3c(trace, line_count, target_ways):
    """Compare the target with an equal-size fully associative LRU cache."""
    target = LRUBlockCache(line_count, target_ways)
    fully_associative = LRUBlockCache(line_count, line_count)
    seen = set()
    counts = {"compulsory": 0, "capacity": 0, "conflict": 0}

    for block in trace:
        target_hit = target.access(block)
        full_hit = fully_associative.access(block)

        if not target_hit:
            if block not in seen:
                counts["compulsory"] += 1
            elif not full_hit:
                counts["capacity"] += 1
            else:
                counts["conflict"] += 1
        seen.add(block)

    return counts


trace = [0, 4, 0, 1, 2, 3, 4, 0]
counts = classify_3c(trace, line_count=4, target_ways=1)
assert counts == {"compulsory": 5, "capacity": 2, "conflict": 1}
print(counts)
```

</details>

Multicore coherence introduces additional misses when another core invalidates a line; that fourth cause belongs with Chapter 11. The 3C model here describes a single request stream without coherence invalidations.

### **Average Memory Access Time** {#average-memory-access-time}

**Average memory access time (AMAT)** combines the fast hit path with the slower miss path. When miss penalty means additional time after the cache hit check,

$$
AMAT=T_{hit}+r_m\times P_m.
$$

- $T_{hit}$ is the time paid by every access to select the set, compare tags, and return data on a hit.
- $r_m$ is miss rate, the fraction of accesses that miss this cache.
- $P_m$ is the average additional miss penalty after discovering the miss.

![Every request pays hit time. A fraction equal to the miss rate additionally pays the lower-level miss penalty before the pipeline can use the data.](assets/amat-access-tree.svg){fig-align="center" width="100%"}

For a 1 ns hit time, 5% miss rate, and 60 ns additional penalty,

$$
AMAT=1+0.05\times60=4\text{ ns}.
$$

Although 95% of accesses hit in one nanosecond, the rare long path contributes three extra nanoseconds to the average. This shows why a small miss-rate change can matter when the miss penalty is large.

An alternative convention defines $T_{miss,total}$ as the entire time of a miss, including the initial cache check:

$$
AMAT=(1-r_m)T_{hit}+r_mT_{miss,total}.
$$

Both equations agree when definitions are consistent. Mixing a total miss time into the first equation double-counts or omits work.

<details>
<summary>Python example: connect AMAT to pipeline CPI</summary>

```python
def amat(hit_time, miss_rate, additional_miss_penalty):
    return hit_time + miss_rate * additional_miss_penalty


def memory_stall_cpi(
    base_cpi,
    instruction_miss_rate,
    instruction_penalty_cycles,
    data_accesses_per_instruction,
    data_miss_rate,
    data_penalty_cycles,
):
    """First-order CPI model assuming the miss penalties do not overlap."""
    instruction_stalls = instruction_miss_rate * instruction_penalty_cycles
    data_stalls = data_accesses_per_instruction * data_miss_rate * data_penalty_cycles
    return base_cpi + instruction_stalls + data_stalls


assert amat(1.0, 0.05, 60.0) == 4.0

cpi = memory_stall_cpi(
    base_cpi=1.0,
    instruction_miss_rate=0.01,
    instruction_penalty_cycles=40,
    data_accesses_per_instruction=0.30,
    data_miss_rate=0.04,
    data_penalty_cycles=50,
)

assert round(cpi, 2) == 2.00
print(f"AMAT = {amat(1.0, 0.05, 60.0):.2f} ns; CPI = {cpi:.2f}")
```

</details>

The CPI equation connects directly to Chapter 07: an instruction-cache miss starves IF, while a load miss delays the value needed by dependent instructions. Modern processors can overlap misses and continue independent work, so the additive model is a first-order bound rather than a cycle-accurate description.

### **Multi-Level Caches** {#multi-level-caches}

One cache must balance a short hit time against a low miss rate. A **multi-level hierarchy** uses a small L1 for the shortest common path and larger lower caches to intercept misses before they reach DRAM. The L1 is often split so IF and MEM can access instruction and data arrays concurrently, eliminating the structural conflict introduced in Chapter 07.

::: {.diagram-scroll .wide-diagram}
![Instruction fetch checks L1I and data access checks L1D. Misses continue to L2, the last-level cache, and DRAM; returned blocks refill the path to the requester.](assets/multi-level-cache-path.svg){fig-align="center"}
:::

For two cache levels, using a **local** L2 miss rate measured only among L1 misses,

$$
AMAT=T_{L1}+MR_{L1}\left(T_{L2}+MR_{L2|L1miss}\times P_{mem}\right).
$$

- $T_{L1}$ is paid for every request.
- $MR_{L1}$ is the fraction of all requests that reach L2.
- $T_{L2}$ is the additional L2 lookup time after an L1 miss.
- $MR_{L2|L1miss}$ is the fraction of L2 lookups that also miss.
- $P_{mem}$ is the additional lower-memory penalty after the L2 miss.

The global fraction of all requests reaching memory is

$$
MR_{L2,global}=MR_{L1}\times MR_{L2|L1miss}.
$$

Confusing local and global rates is a common source of incorrect AMAT calculations.

<details>
<summary>Python example: recursively calculate a multi-level AMAT</summary>

```python
def multilevel_amat(l1_time, l1_miss_rate, l2_time, local_l2_miss_rate, memory_penalty):
    """All lower times are additional after the preceding miss is known."""
    l1_miss_service = l2_time + local_l2_miss_rate * memory_penalty
    average = l1_time + l1_miss_rate * l1_miss_service
    global_memory_rate = l1_miss_rate * local_l2_miss_rate
    return average, global_memory_rate


average, memory_rate = multilevel_amat(
    l1_time=1.0,
    l1_miss_rate=0.08,
    l2_time=4.0,
    local_l2_miss_rate=0.15,
    memory_penalty=80.0,
)

assert round(average, 2) == 2.28
assert round(memory_rate, 3) == 0.012
print(f"AMAT={average:.2f} ns, requests reaching memory={memory_rate:.1%}")
```

</details>

Levels also need a relationship between copies. An inclusive hierarchy guarantees that an upper-level block also appears below, an exclusive hierarchy uses levels for disjoint capacity, and a non-inclusive/non-exclusive design provides no simple guarantee. These choices affect eviction, coherence, and usable capacity but do not change the basic lookup dependency.

On a miss, refill may begin with the critical requested bytes so the pipeline can resume before the entire block arrives. Non-blocking caches can track outstanding misses and continue serving independent hits. Such mechanisms reduce effective penalty by overlapping work; they do not convert a miss into a hit.

### **Cache-Aware Program Behavior** {#cache-aware-program-behavior}

The cache is usually transparent, but software controls the address sequence. Two implementations with the same arithmetic complexity can produce radically different block reuse. Cache-aware programming improves locality without depending on the exact contents of the cache at any instant.

::: {.diagram-scroll .wide-diagram}
![For a row-major matrix, row traversal consumes neighboring elements, column traversal uses a large stride, and tiling keeps a small two-dimensional region active long enough for reuse.](assets/cache-aware-access-patterns.svg){fig-align="center"}
:::

Useful transformations include:

| Transformation | Locality effect | Important caution |
|---|---|---|
| traverse in storage order | consumes neighboring elements from each block | language/layout determines the contiguous dimension |
| loop tiling or blocking | reduces reuse distance by finishing work on a small region | tile must share cache capacity with other active data |
| compact related fields | places jointly used bytes in fewer blocks | unused fields can waste bandwidth |
| structure of arrays | streams only fields used by a vectorized operation | may hurt code that needs whole records together |
| align or pad objects | can remove pathological set conflicts | extra padding consumes capacity |
| prefetch | overlaps a predictable compulsory transfer | too early/unused prefetches waste capacity and bandwidth |

<details>
<summary>Python model: count misses for row, column, and tiled traversal</summary>

```python
class DirectMappedByteCache:
    def __init__(self, capacity_bytes, block_bytes):
        self.block_bytes = block_bytes
        self.tags = [None] * (capacity_bytes // block_bytes)
        self.misses = 0

    def access(self, address):
        block = address // self.block_bytes
        line = block % len(self.tags)
        tag = block // len(self.tags)
        if self.tags[line] != tag:
            self.misses += 1
            self.tags[line] = tag


def matrix_addresses(n, order, tile=8, element_bytes=8):
    def address(i, j):
        return (i * n + j) * element_bytes  # row-major layout

    if order == "row":
        for i in range(n):
            for j in range(n):
                yield address(i, j)
    elif order == "column":
        for j in range(n):
            for i in range(n):
                yield address(i, j)
    elif order == "tiled-column":
        # Preserve column-oriented work, but complete one small tile at a time.
        for jj in range(0, n, tile):
            for ii in range(0, n, tile):
                for j in range(jj, min(jj + tile, n)):
                    for i in range(ii, min(ii + tile, n)):
                        yield address(i, j)
    else:
        raise ValueError("unknown order")


def miss_count(order):
    cache = DirectMappedByteCache(capacity_bytes=4 * 1024, block_bytes=64)
    for address in matrix_addresses(64, order):
        cache.access(address)
    return cache.misses


misses = {order: miss_count(order) for order in ["row", "column", "tiled-column"]}
assert misses == {"row": 512, "column": 4096, "tiled-column": 512}
print(misses)
```

</details>

The model deliberately omits associativity, prefetching, multiple levels, and overlap so the access-order effect is visible. Real measurements should use representative input sizes and hardware counters rather than assuming every machine has the same cache geometry.

Cache-aware behavior should preserve correctness and maintainability first. Optimizing a hot loop after measurement is more valuable than globally complicating data layout without evidence. The durable lesson is to reason in blocks, working sets, reuse distance, and access order.

**Chapter summary.** A memory hierarchy combines small fast upper levels with larger slower lower levels because no one technology provides minimum latency, maximum capacity, bandwidth, and low cost. Temporal locality rewards retaining recently used blocks, while spatial locality rewards transferring nearby bytes together. A cache decomposes each address into tag, set index, and block offset; valid tags determine hits, the index limits placement, and the offset selects bytes. Direct mapping minimizes lookup hardware, set associativity balances conflicts and cost, and full associativity maximizes placement freedom. Replacement policies predict which resident block is least useful. Write-through versus write-back determines when updates propagate, while write-allocate versus no-write-allocate determines behavior on a store miss. The 3C model separates compulsory, capacity, and conflict misses. AMAT weights hit time and miss penalty, and memory stalls connect the hierarchy directly to pipeline CPI. Multiple cache levels protect the L1 hit path while filtering DRAM traffic. Finally, traversal order, tiling, layout, alignment, and prefetching can improve locality without changing the program's mathematical result. Chapter 09 will add virtual addresses, page tables, and TLBs, then join translation with the cache lookup path developed here.